# exp-00 Where a turn's time and tokens go

**Question.** Of a turn's latency and input tokens, how much is STT, each LLM step, tools and the follow-up chips
call; how many LLM steps does a turn take, and what does the first step do?

**Data.** `traces.db`: real app sessions only (12-hex session ids), prompt v20 or later, turns ≥ 100 ms.
Free: no LLM calls. Code: `analyze.py`.

In [ ]:
# Setup: works whether the notebook's folder or the repo root is the working directory.
import json, sys
from pathlib import Path
HERE = Path.cwd()
if HERE.name != "exp00_baseline":
    HERE = next(p for p in [Path.cwd() / "eng-experiments" / "exp00_baseline", Path.cwd() / "exp00_baseline"] if p.exists())
sys.path[:0] = [str(HERE), str(HERE.parent)]
import os; os.environ.setdefault("TRACE_PRINT", "0")   # no per-span printing during runs

import pandas as pd
from IPython.display import Markdown, display
import common
import analyze
pd.set_option("display.max_colwidth", 120)

## Load the real turns

In [ ]:
spans = analyze.load()
turns = analyze.turns_of(spans)
print(f"{len(turns)} turns from {len({t['session'] for t in turns})} sessions")
pd.DataFrame([{"model": t["model"], "prompt": t["prompt"], "steps": len(t["steps"]), "turn_s": t["ms"] / 1000,
               "llm_s": t["llm_ms"] / 1000, "tool_s": t["tool_ms"] / 1000, "text": t["text"]} for t in turns]).tail(10)

## Turn latency by model

In [ ]:
lat = analyze.latency_table(turns)
pd.DataFrame([{"model": m, "turns": v["turns"], "steps_mean": v["steps_mean"], "steps_hist": v["steps_hist"],
               "turn_s_p50": v["turn_s"]["p50"], "turn_s_p90": v["turn_s"]["p90"], "turn_s_p95": v["turn_s"]["p95"],
               "turn_s_p99_or_max": v["turn_s"].get("p99", v["turn_s"].get("max")),
               "llm_step_ms_p50": v["llm_step_ms"]["p50"], "share_llm": v["share_llm"],
               "share_tools": v["share_tools"], "share_other": v["share_other"]} for m, v in lat.items()]).set_index("model")

## What the first LLM step does (share of turns)

In [ ]:
pd.DataFrame(analyze.first_step_table(turns)).T.fillna(0)

## Edit calls per step

In [ ]:
pd.DataFrame(analyze.edits_per_step(turns)).T

## Tool time

In [ ]:
pd.DataFrame(analyze.tool_time(turns)).T

## STT and the follow-up chips call

In [ ]:
pd.DataFrame(analyze.stt_and_chips(spans)).T

## Input tokens of one LLM step (o200k)

In [ ]:
tok = analyze.tokens_table(turns)
display(pd.Series(tok["o200k"], name="o200k tokens").to_frame())
pd.DataFrame(tok["provider_reported_step1"]).T

## Save `results/exp00.json` and regenerate `README.md`

In [ ]:
analyze.main()